# Bayesian Spatial Flow (Origin–Destination) Models

This notebook demonstrates the **cross-sectional** flow models in `bayespecon`:

| Model | Description |
|---|---|
| `SARFlow` | Three free ρ parameters — destination, origin, network |
| `SARFlowSeparable` | Constrained ρ_w = −ρ_d·ρ_o; exact eigenvalue log-det |

Both are fully Bayesian implementations of the LeSage–Fischer (2008) SAR flow framework.

---

## Background: The SAR Flow Model

Let $y$ be the $N$-vector of observed flows ($N = n^2$ O-D pairs). The model is

$$
y = \rho_d W_d y + \rho_o W_o y + \rho_w W_w y + X\beta + \varepsilon,
\quad \varepsilon \sim \mathcal{N}(0, \sigma^2 I_N)
$$

where

$$
W_d = I_n \otimes W, \quad W_o = W \otimes I_n, \quad W_w = W \otimes W
$$

capture **destination**, **origin**, and **network** (O-D pair) neighbourhood dependence respectively.

The **reduced form** is

$$
y = (I_N - \rho_d W_d - \rho_o W_o - \rho_w W_w)^{-1}(X\beta + \varepsilon)
$$

The design matrix $X$ follows the LeSage layout with separate destination, origin, and intra-zonal coefficient blocks:

$$
\beta = [\alpha,\; \alpha_i,\; \beta_d^1\ldots\beta_d^k,\;
         \beta_o^1\ldots\beta_o^k,\; \beta_i^1\ldots\beta_i^k]
$$

### Separable variant

`SARFlowSeparable` imposes $\rho_w = -\rho_d \rho_o$, which makes the filter matrix factorisable as a Kronecker product:

$$
A = (I_n - \rho_d W) \otimes (I_n - \rho_o W)
\implies \log|A| = n\log|I_n - \rho_d W| + n\log|I_n - \rho_o W|
$$

This enables **exact, O(n) log-det** evaluation via eigenvalues of the small $n\times n$ matrix $W$.

## Setup

In [ ]:
import warnings

import arviz as az
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from libpysal.graph import Graph

from bayespecon import SARFlow, SARFlowSeparable, flow_design_matrix
from bayespecon.dgp.flows import generate_flow_data, generate_flow_data_separable

warnings.filterwarnings("ignore")
az.style.use("arviz-white")
rng = np.random.default_rng(42)

## 1  Simulated Data

We simulate flow data from a ring-contiguity graph on $n = 12$ spatial units, giving $N = 144$ O-D pairs.

**True parameters**

| Parameter | Value |
|-----------|-------|
| ρ_d | 0.35 |
| ρ_o | 0.25 |
| ρ_w | 0.10 |
| β_d | [1.0, −0.5] |
| β_o | [0.5, 0.3] |
| σ | 1.0 |

In [ ]:
def make_ring_graph(n: int) -> Graph:
    """Row-standardised ring-contiguity Graph on n units."""
    focal = np.concatenate([np.arange(n), np.arange(n)])
    neighbor = np.concatenate([np.roll(np.arange(n), 1), np.roll(np.arange(n), -1)])
    weight = np.ones(len(focal), dtype=float)
    G = Graph.from_arrays(focal, neighbor, weight)
    return G.transform("r")


# --- True parameters --------------------------------------------------------
n = 12
N = n * n

RHO_D = 0.35
RHO_O = 0.25
RHO_W = 0.10
BETA_D = np.array([1.0, -0.5])
BETA_O = np.array([0.5, 0.3])
SIGMA = 1.0

# --- Graph and DGP ----------------------------------------------------------
G = make_ring_graph(n)

data = generate_flow_data(
    n,
    G,
    rho_d=RHO_D,
    rho_o=RHO_O,
    rho_w=RHO_W,
    beta_d=BETA_D,
    beta_o=BETA_O,
    sigma=SIGMA,
    seed=42,
)

y_vec = data["y_vec"]  # (N,)  vectorised O-D flows
X = data["X"]  # (N, p) design matrix (dest/orig/intra blocks + intercept)
col_names = data["col_names"]

print(f"Flow observations: N = {N}  ({n}×{n})")
print(f"Design matrix shape: {X.shape}")
print(f"Column names: {col_names}")
print(
    f"\ny summary:  min={y_vec.min():.2f}  mean={y_vec.mean():.2f}  max={y_vec.max():.2f}"
)

In [ ]:
# Visualise the flow matrix
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

im = axes[0].imshow(y_vec.reshape(n, n), cmap="YlOrRd")
axes[0].set_title("Simulated flow matrix $Y$ (origin × destination)")
axes[0].set_xlabel("Destination")
axes[0].set_ylabel("Origin")
plt.colorbar(im, ax=axes[0])

axes[1].hist(y_vec, bins=30, edgecolor="white")
axes[1].set_title("Distribution of $y_{ij}$ values")
axes[1].set_xlabel("Flow")
axes[1].set_ylabel("Count")

# plt.tight_layout()
plt.show()

## 2  `SARFlow`: Three-parameter model

`SARFlow` places a Dirichlet prior on $(ρ_d, ρ_o, ρ_w)$ that enforces positivity and the stability constraint $\rho_d + \rho_o + \rho_w \leq 1$ exactly, using the stick-breaking transformation for NUTS efficiency.

The log-determinant $\log|A|$ is evaluated via the Barry–Pace trace-stochastic method, which requires only sparse matrix–vector products.

In [ ]:
sar_flow = SARFlow(
    y_vec,
    G,
    X,
    col_names=col_names,
    logdet_method="traces",  # Barry-Pace stochastic trace (default)
    restrict_positive=True,  # Dirichlet stability prior
    miter=20,  # trace polynomial order (increase for better accuracy)
    trace_seed=0,
)

idata_sar = sar_flow.fit(
    draws=1000,
    tune=1000,
    chains=4,
    target_accept=0.9,
    random_seed=42,
    progressbar=True,
)

In [ ]:
# Posterior summary for the spatial autoregressive parameters
summary_rho = sar_flow.summary(var_names=["rho_d", "rho_o", "rho_w"])
print("=== SARFlow: spatial autoregressive parameters ===")
print(summary_rho.to_string())

# Compare to true values
print(f"\nTrue values:  rho_d={RHO_D}  rho_o={RHO_O}  rho_w={RHO_W}")

In [ ]:
# Full coefficient summary
summary_beta = sar_flow.summary(var_names=["beta", "sigma"])
print("=== SARFlow: regression coefficients ===")
print(summary_beta.to_string())

In [ ]:
# Trace plots for the three ρ parameters
az.plot_trace(
    idata_sar,
    var_names=["rho_d", "rho_o", "rho_w"],
    figsize=(10, 6),
)
plt.suptitle("SARFlow — posterior traces", y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Posterior densities with true-value markers
fig, axes = plt.subplots(1, 3, figsize=(12, 3))
params = [("rho_d", RHO_D), ("rho_o", RHO_O), ("rho_w", RHO_W)]

for ax, (param, true_val) in zip(axes, params):
    az.plot_posterior(
        idata_sar,
        var_names=[param],
        ax=ax,
        ref_val=true_val,
        hdi_prob=0.94,
    )
    # ax.set_title(f"${param.replace('_', r'\\_')}$ (true = {true_val})")

plt.suptitle("SARFlow — posterior distributions (red line = true value)", y=1.02)
plt.tight_layout()
plt.show()

## 3  `SARFlowSeparable`: Constrained model

`SARFlowSeparable` imposes the constraint $\rho_w = -\rho_d \rho_o$, yielding:

* **Exact log-det** via eigenvalues of the small $n \times n$ matrix $W$ — no Monte Carlo traces needed.
* **Two free parameters** ($\rho_d$, $\rho_o$) instead of three.
* Typically **faster mixing** because the posterior geometry is simpler.

We generate new data that satisfies the constraint ($\rho_w = -\rho_d\rho_o = -0.12$) to give the model the best chance at recovery.

In [ ]:
# True parameters for the separable model (asymmetric for identifiability)
RHO_D_SEP = 0.40
RHO_O_SEP = 0.30
RHO_W_SEP = -RHO_D_SEP * RHO_O_SEP  # = -0.12 (constraint)

print(
    f"Separable true values:  rho_d={RHO_D_SEP}  rho_o={RHO_O_SEP}"
    f"  rho_w=-rho_d*rho_o={RHO_W_SEP:.4f}"
)

data_sep = generate_flow_data_separable(
    n,
    G,
    rho_d=RHO_D_SEP,
    rho_o=RHO_O_SEP,
    beta_d=BETA_D,
    beta_o=BETA_O,
    sigma=SIGMA,
    seed=7,
)

y_sep = data_sep["y_vec"]
X_sep = data_sep["X"]
cn_sep = data_sep["col_names"]

print(
    f"\ny_sep summary:  min={y_sep.min():.2f}  mean={y_sep.mean():.2f}  max={y_sep.max():.2f}"
)

In [ ]:
sep_flow = SARFlowSeparable(
    y_sep,
    G,
    X_sep,
    col_names=cn_sep,
    # logdet_method="eigenvalue" is the default for SARFlowSeparable
    trace_seed=0,
)

idata_sep = sep_flow.fit(
    draws=1000,
    tune=1000,
    chains=4,
    target_accept=0.9,
    random_seed=42,
    progressbar=True,
)

In [ ]:
summary_sep = sep_flow.summary(var_names=["rho_d", "rho_o"])
print("=== SARFlowSeparable: spatial autoregressive parameters ===")
print(summary_sep.to_string())
print(
    f"\nTrue values:  rho_d={RHO_D_SEP}  rho_o={RHO_O_SEP}"
    f"  (rho_w = -rho_d*rho_o = {RHO_W_SEP:.4f} is derived)"
)

In [ ]:
az.plot_trace(
    idata_sep,
    var_names=["rho_d", "rho_o"],
    figsize=(10, 4),
)
plt.suptitle("SARFlowSeparable — posterior traces", y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(8, 3))
for ax, (param, true_val) in zip(axes, [("rho_d", RHO_D_SEP), ("rho_o", RHO_O_SEP)]):
    az.plot_posterior(
        idata_sep, var_names=[param], ax=ax, ref_val=true_val, hdi_prob=0.94
    )
    ax.set_title(f"${param.replace('_', r'_')}$ (true = {true_val})")

plt.suptitle("SARFlowSeparable — posterior distributions", y=1.02)
plt.tight_layout()
plt.show()

## 4  Design Matrix Details

The `flow_design_matrix` helper builds the standard LeSage O-D design matrix with destination, origin, and intra-zonal coefficient blocks.  Each column of the regional attribute matrix $X$ (shape $n \times k$) produces **three** columns in the full design matrix: one for destination effects, one for origin effects, and one for intra-zonal effects.

| Column group | Construction | Interpretation |
|---|---|---|
| intercept | $\mathbf{1}_N$ | global mean flow |
| intra_indicator | $\text{vec}(I_n)$ | intra-zonal dummy |
| dest_* | $\iota_n \otimes X$ | destination characteristics |
| orig_* | $X \otimes \iota_n$ | origin characteristics |
| intra_* | $\text{diag}(I_n) \cdot (I_n \otimes X)$ | intra-zonal attr. |

You can also pass a pre-built `pd.DataFrame` directly as `X` — column names are inferred automatically.

In [ ]:
# Build a design matrix from scratch using regional attributes
X_regional = rng.standard_normal((n, 2))  # n × k attribute matrix
dm = flow_design_matrix(X_regional, col_names=["income", "pop"])

print(f"Regional attribute matrix: {X_regional.shape}  (n × k)")
print(f"Full O-D design matrix:    {dm.combined.shape}  (N × p)")
print(f"\nColumn names:\n  {dm.feature_names}")

pd.DataFrame(dm.combined[:6], columns=dm.feature_names).round(3)

## 5  Spatial Effects

The flow SAR model has a rich effects decomposition.  For a one-unit shock to predictor $p$ in all regions, the total change in aggregate flows is:

$$
T_p = \iota_N' A^{-1} (\beta_d^{(p)} + \beta_o^{(p)}) \iota_N / N
$$

where $A = I_N - \rho_d W_d - \rho_o W_o - \rho_w W_w$.

`_compute_spatial_effects_posterior()` returns posterior draws of origin, destination, intra, network, and total effects for each predictor.  Below we summarise them as posterior means with 94% credible intervals.

In [ ]:
effects = sar_flow._compute_spatial_effects_posterior(draws=500)

# k = number of regional predictors; col_names without intercept/intra blocks
k = sar_flow._k
pred_names = [c.replace("dest_", "") for c in col_names if c.startswith("dest_")]

rows = []
for effect_name, draws_arr in effects.items():
    for j, pred in enumerate(pred_names):
        col = draws_arr[:, j]
        rows.append(
            {
                "effect": effect_name,
                "predictor": pred,
                "mean": col.mean(),
                "hdi_3%": np.percentile(col, 3),
                "hdi_97%": np.percentile(col, 97),
            }
        )

effects_df = pd.DataFrame(rows)
print(effects_df.pivot(index="predictor", columns="effect", values="mean").round(4))

## 6  MCMC Diagnostics

Good practice after any Bayesian fit: check $\hat{R}$ (should be $< 1.01$) and effective sample size (ESS > 400 per chain).

In [ ]:
for label, idata in [("SARFlow", idata_sar), ("SARFlowSeparable", idata_sep)]:
    diag = az.summary(idata, var_names=["rho_d", "rho_o"], stat_focus="mean")
    rhat_ok = (diag["r_hat"] < 1.01).all()
    ess_ok = (diag["ess_bulk"] > 400).all()
    print(f"{label:25s}  r_hat<1.01: {rhat_ok}   ess_bulk>400: {ess_ok}")
    print(diag[["mean", "sd", "hdi_3%", "hdi_97%", "ess_bulk", "r_hat"]].to_string())
    print()

In [ ]:
# Energy plot — checks that HMC is exploring the posterior efficiently
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
az.plot_energy(idata_sar, ax=axes[0])
axes[0].set_title("SARFlow energy")
az.plot_energy(idata_sep, ax=axes[1])
axes[1].set_title("SARFlowSeparable energy")
plt.tight_layout()
plt.show()

## 7  Model Comparison

The WAIC / LOO-CV scores can be used to compare `SARFlow` and `SARFlowSeparable` when both are estimated on the same data.  A lower ELPD (expected log pointwise predictive density) indicates a worse-fitting model.

> **Note**: here we fit `SARFlow` to the separable data to make the comparison meaningful — the separable model is nested in the unrestricted one.

In [ ]:
# Fit unrestricted SARFlow on the same separable data for a fair comparison
sar_flow_on_sep = SARFlow(
    y_sep,
    G,
    X_sep,
    col_names=cn_sep,
    logdet_method="traces",
    miter=20,
    trace_seed=0,
)
idata_sar_on_sep = sar_flow_on_sep.fit(
    draws=1000,
    tune=1000,
    chains=4,
    target_accept=0.9,
    random_seed=42,
    progressbar=True,
)

In [ ]:
# LOO-CV comparison (requires log-likelihood in idata; falls back to WAIC if unavailable)
try:
    loo_sar = az.loo(idata_sar_on_sep)
    loo_sep = az.loo(idata_sep)
    comparison = az.compare(
        {"SARFlow (3 ρ)": idata_sar_on_sep, "SARFlowSeparable": idata_sep}
    )
    print("LOO-CV model comparison:")
    print(
        comparison[
            ["elpd_loo", "p_loo", "elpd_diff", "weight", "se", "warning"]
        ].to_string()
    )
except Exception as e:
    print(f"LOO not available (log-likelihood not stored): {e}")
    print("Tip: pass compute_log_likelihood=True to fit() to enable LOO/WAIC.")

## 8  Prior Sensitivity and Stability Constraint

By default `SARFlow` uses a Dirichlet prior that forces $\rho_d, \rho_o, \rho_w \geq 0$ and $\rho_d + \rho_o + \rho_w \leq 1$.  Setting `restrict_positive=False` allows negative spillovers via independent `Uniform(-1, 1)` priors plus a differentiable stability wall potential.

Use `restrict_positive=False` when competitive effects (e.g. negative network parameter) are theoretically expected.

In [ ]:
# Fit with restrict_positive=False to allow negative rho values
sar_flow_neg = SARFlow(
    y_vec,
    G,
    X,
    col_names=col_names,
    logdet_method="traces",
    restrict_positive=False,  # Uniform(-1,1) priors + stability potential
    miter=20,
    trace_seed=0,
)
idata_neg = sar_flow_neg.fit(
    draws=800,
    tune=1000,
    chains=4,
    target_accept=0.95,
    random_seed=42,
    progressbar=True,
)

summary_neg = sar_flow_neg.summary()
print("=== SARFlow (restrict_positive=False) ===")
print(summary_neg[["mean", "sd", "hdi_3%", "hdi_97%", "r_hat"]].to_string())

## 9  Quick Reference

```python
from bayespecon import SARFlow, SARFlowSeparable
from bayespecon import flow_design_matrix
from bayespecon.dgp.flows import generate_flow_data, generate_flow_data_separable

# Build design matrix from regional attributes
dm = flow_design_matrix(X_regional, col_names=["income", "pop"])

# --- SARFlow (three free ρ parameters) ---
model = SARFlow(
    y, G, dm.combined,
    col_names=dm.feature_names,
    logdet_method="traces",      # Barry-Pace stochastic traces (default)
    restrict_positive=True,      # Dirichlet stability prior
)
idata = model.fit(draws=2000, tune=1000, chains=4, random_seed=0)
model.summary(var_names=["rho_d", "rho_o", "rho_w"])

# --- SARFlowSeparable (ρ_w = -ρ_d·ρ_o, exact log-det) ---
sep_model = SARFlowSeparable(
    y, G, dm.combined,
    col_names=dm.feature_names,
    # logdet_method="eigenvalue" is the default
)
idata_sep = sep_model.fit(draws=2000, tune=1000, chains=4, random_seed=0)
```

### Key parameters

| Parameter | Default | Description |
|---|---|---|
| `logdet_method` | `"traces"` | `"traces"` (Barry–Pace) or `"eigenvalue"`/`"chebyshev"` for separable |
| `restrict_positive` | `True` | Dirichlet prior; set `False` for negative-spillover models |
| `miter` | 30 | Trace polynomial order (higher = more accurate, slower precomputation) |
| `trace_riter` | 50 | Monte Carlo probes for trace estimation |

### Model choice guide

| Data characteristic | Recommended model |
|---|---|
| No prior on sign of spatial effects | `SARFlow(restrict_positive=False)` |
| Positive spillovers expected | `SARFlow(restrict_positive=True)` |
| Separability plausible, fast inference needed | `SARFlowSeparable` |
| Count/non-negative flows | `PoissonSARFlow` or `PoissonSARFlowSeparable` |